# Getting the train/test split
## Strategy Used:
To keep in the mind the nature of the time series dataset, we can't do random sampling. Instead, we used a modulo splitting strategy to split the data into an 80/10/10 train-val-test split based on the day of the year (thus keeping hourly data together)

In [1]:
import pandas as pd

bike_df_root = pd.read_csv('final_bike_demand_dataset.csv')
bike_df = bike_df_root.copy()

In [2]:
bike_df.columns

Index(['datetime', 'station_id', 'date', 'hour', 'day_of_week', 'month',
       'year', 'is_weekend', 'is_am_rush_hour', 'is_pm_rush_hour',
       'trips_started', 'trips_ended', 'net_flow', 'temp_c', 'dew_point_c',
       'rel_humidity_pct', 'precip_mm', 'wind_speed_kmh', 'visibility_km',
       'pressure_kpa', 'station_name', 'latitude', 'longitude', 'capacity',
       'is_charging_station', 'bus', 'subway', 'streetcar', 'train',
       'public_transport_total', 'tourism', 'office', 'park', 'healthcare',
       'education', 'restaurants_bars', 'commercial', 'cluster_east_end',
       'cluster_scarborough', 'cluster_uptown', 'cluster_west_end',
       'dist_to_union_km', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'month_sin', 'month_cos'],
      dtype='object')

### Perform Train/Val/Test Split

In [3]:
# Ensure datetime is actually a datetime object
bike_df['datetime'] = pd.to_datetime(bike_df['datetime'])

# Create day_of_year column
bike_df['day_of_year'] = bike_df['datetime'].dt.dayofyear

# Define Masks using Modulo 10
# Test: Day % 10 == 0 (10%)
mask_test = (bike_df['day_of_year'] % 10 == 0)

# Validation: Day % 10 == 1 (10%)
mask_val = (bike_df['day_of_year'] % 10 == 1)

# Train: Everything else (80%)
mask_train = ~(mask_test | mask_val)

# Apply Splits
test_df = bike_df[mask_test].copy()
val_df = bike_df[mask_val].copy()
train_df = bike_df[mask_train].copy()

print(f"Full Train Set:      {train_df.shape} ({len(train_df)/len(bike_df):.1%})")
print(f"Full Validation Set: {val_df.shape}   ({len(val_df)/len(bike_df):.1%})")
print(f"Full Test Set:       {test_df.shape}  ({len(test_df)/len(bike_df):.1%})")

Full Train Set:      (10865904, 49) (80.0%)
Full Validation Set: (1381274, 49)   (10.2%)
Full Test Set:       (1338746, 49)  (9.9%)


### Negative Downsampling 
Applied only to training set to balance out the heavy bias towards zero at the moment

In [4]:
print("\n--- Starting Negative Sampling on Training Set ---")

# 1. Separate Positives and Zeros
positives = train_df[train_df['trips_started'] > 0]
zeros = train_df[train_df['trips_started'] == 0]

print(f"Positive Rows: {len(positives):,}")
print(f"Zero Rows:     {len(zeros):,}")

# 2. Random Downsampling
# Target Ratio: Keep 3 Zero rows for every 1 Positive row
target_zeros = len(positives) * 3

if len(zeros) > target_zeros:
    print(f"Downsampling zeros to {target_zeros:,} (Ratio 1:3)...")
    zeros_sampled = zeros.sample(n=target_zeros, random_state=42)
    
    # Recombine and Shuffle
    train_balanced = pd.concat([positives, zeros_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)
else:
    print("Warning: Not enough zeros to reach target ratio. Keeping all zeros.")
    train_balanced = train_df.copy()

print(f"Balanced Train Shape: {train_balanced.shape}")


--- Starting Negative Sampling on Training Set ---
Positive Rows: 3,169,513
Zero Rows:     7,696,391
Balanced Train Shape: (10865904, 49)


### Get Tuning Sample
#### This is what will be used for most of our experiments

In [5]:
SAMPLE_FRACTION = 0.10 
train_sample = train_balanced.sample(frac=0.10, random_state=42)
print(f"Tuning Sample:  {train_sample.shape}")

Tuning Sample:  (1086590, 49)


### Export Train/Test/Val Split and Sample

In [6]:
# 1. Export the BALANCED Training Set (For the Model)
train_balanced.to_csv('train_2.csv', index=False)

# 2. Export the FULL Validation & Test Sets (For Evaluation)
val_df.to_csv('val_2.csv', index=False)
test_df.to_csv('test_2.csv', index=False)

# 3. Export a Small Sample of Balanced Train (For Fast Tuning)
train_sample = train_balanced.sample(frac=0.10, random_state=42)
train_sample.to_csv('train_sample_2.csv', index=False)

print("\n--- Exports Complete ---")
print("1. train_2.csv        (Use for Training)")
print("2. val_2.csv                 (Use for Tuning/Early Stopping)")
print("3. test_2.csv                (Use for Final Score)")
print("4. train_sample_2.csv (Use for Quick Experiments)")


--- Exports Complete ---
1. train_2.csv        (Use for Training)
2. val_2.csv                 (Use for Tuning/Early Stopping)
3. test_2.csv                (Use for Final Score)
4. train_sample_2.csv (Use for Quick Experiments)
